<a href="https://colab.research.google.com/github/shreyasat27/mahework2025/blob/main/Criteria_1(well%20defined%20state).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Complete one file to prove DiVincenzo's Criteria for "Well defined two level system"**

In [2]:
!pip install pyscf
!pip install rdkit
!pip install qutip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.6/31.6 MB 52.6 MB/s eta 0:00:00


In [3]:
from pyscf import gto, scf
import numpy as np

def analyze_molecule(atom_geometry, basis="sto-3g", charge=0, spin=0):
    """
    Performs a preliminary quantum chemical analysis to assess a molecule's
    potential as a spin qubit candidate based on spin state and HOMO-LUMO gap.

    Args:
        atom_geometry (str): A string defining the molecular geometry (e.g., "C 0 0 0; H 0 0 1").
        basis (str): The atomic basis set to use for the calculation (default: "sto-3g").
        charge (int): The total charge of the molecule.
        spin (int): The number of unpaired electrons (2S, where S is total spin).

    Returns:
        dict: A dictionary containing key electronic structure properties.
    """

    # 1. MOLECULE DEFINITION
    # Initialize the molecule object and set its properties
    mol = gto.Mole()
    mol.atom = atom_geometry  # Set atomic coordinates
    mol.basis = basis         # Set the basis set
    mol.charge = charge       # Set molecular charge
    mol.spin = spin           # Set number of unpaired electrons (2S)
    mol.build()               # Finalize the molecule construction

    # 2. CHOOSE CALCULATION METHOD
    # Use Restricted (RHF) for closed-shell, Unrestricted (UHF) for open-shell
    if spin == 0:
        mf = scf.RHF(mol)  # Restricted Hartree-Fock
    else:
        mf = scf.UHF(mol)  # Unrestricted Hartree-Fock

    # 3. CONVERGENCE SETTINGS
    # Techniques to help the Self-Consistent Field (SCF) calculation converge
    mf.max_cycle = 100    # Maximum SCF iterations
    mf.init_guess = 'atom' # Initial electron density guess from atomic orbitals
    mf.damp = 0.5         # Damping factor to mix old and new density matrices
    mf.level_shift = 0.2  # Shifts virtual orbitals to aid convergence

    # 4. RUN THE QUANTUM CHEMISTRY CALCULATION
    energy = mf.kernel()  # Solve the Hartree-Fock equations

    # Check if the SCF calculation converged successfully
    if not mf.converged:
        print("WARNING: SCF did not fully converge!")

    # 5. POST-PROCESSING: EXTRACT KEY PROPERTIES
    mo_energy = mf.mo_energy  # Molecular orbital energies
    # Find HOMO (Highest Occupied Molecular Orbital)
    homo = np.max(mo_energy[mo_energy < 0])
    # Find LUMO (Lowest Unoccupied Molecular Orbital)
    lumo = np.min(mo_energy[mo_energy > 0])
    gap = lumo - homo  # Calculate HOMO-LUMO gap

    # Calculate the total spin expectation value <S^2> and multiplicity
    S2, multiplicity = mf.spin_square()

    # 6. PRINT RESULTS
    print("=== Molecule Analysis ===")
    print(f"Total Energy: {energy:.6f} Hartree")
    print(f"HOMO Energy: {homo:.6f} Ha")
    print(f"LUMO Energy: {lumo:.6f} Ha")
    print(f"HOMO-LUMO Gap: {gap:.6f} Ha")
    print(f"Spin <S^2>: {S2:.4f}")
    print(f"Spin Multiplicity: {multiplicity:.4f}")

    # 7. APPLY QUBIT VIABILITY CRITERIA
    # Strict checks for a potential spin qubit:
    # - Multiplicity ≈ 2 (doublet state, S=1/2)
    # - <S²> ≈ 0.75 (pure spin-1/2, no spin contamination)
    # - Gap > 0 (stable molecule, not a conductor)
    if (abs(multiplicity - 2) < 0.1 and abs(S2 - 0.75) < 0.1 and gap > 0):
        print("\n✅ Candidate qubit system (well-defined two-level + spin-1/2).")
    else:
        print("\n⚠️ Not an ideal qubit candidate (check gap or spin state).")

    # 8. RETURN RESULTS AS A DICTIONARY
    return {
        "energy": energy,
        "homo": homo,
        "lumo": lumo,
        "gap": gap,
        "S2": S2,
        "multiplicity": multiplicity
    }

# Example usage: Analyzing a methyl radical (CH3•)
# Geometry: Carbon at origin, three hydrogens in a trigonal planar arrangement
# Spin = 1 signifies one unpaired electron (a doublet radical)
result = analyze_molecule("C 0 0 0; H 1.0 0 0; H -0.5 0.866 0; H -0.5 -0.866 0", spin=1)

converged SCF energy = -39.0586611617838  <S^2> = 0.76073671  2S+1 = 2.010708
=== Molecule Analysis ===
Total Energy: -39.058661 Hartree
HOMO Energy: -0.368478 Ha
LUMO Energy: 0.320220 Ha
HOMO-LUMO Gap: 0.688698 Ha
Spin <S^2>: 0.7607
Spin Multiplicity: 2.0107

✅ Candidate qubit system (well-defined two-level + spin-1/2).


In [14]:
# Re-run NO with improved settings
result = analyze_molecule("N 0 0 0; O 1.15 0 0", spin=1)

SCF not converged.
SCF energy = -127.526532286614 after 100 cycles  <S^2> = 0.75109292  2S+1 = 2.0010926
=== Molecule Analysis ===
Total Energy: -127.526532 Hartree
HOMO Energy: -0.324102 Ha
LUMO Energy: 0.422171 Ha
HOMO-LUMO Gap: 0.746273 Ha
Spin <S^2>: 0.7511
Spin Multiplicity: 2.0011

✅ Candidate qubit system (well-defined two-level + spin-1/2).


In [15]:
result = analyze_molecule(
    "O 0 0 0; H 0 0 0.96; H 0 0.96 0",  # H₂O geometry (bond angle ~104.5°)
    spin=0,  # Singlet (no unpaired electrons)
    charge=0,
    basis="sto-3g"
)

converged SCF energy = -74.95819976685
=== Molecule Analysis ===
Total Energy: -74.958200 Hartree
HOMO Energy: -0.399682 Ha
LUMO Energy: 0.619709 Ha
HOMO-LUMO Gap: 1.019391 Ha
Spin <S^2>: 0.0000
Spin Multiplicity: 1.0000

⚠️ Not an ideal qubit candidate (check gap or spin state).
